# Notebook 01 — Lane Residue 1

**Repo:** `mod30-residue-lanes`  
**Layer:** `notebooks/`

Notebook 01 begins lane-specific MRL analysis with residue lane `01`.

Constraint view:

> Lane 01 acts as the anchor lane for comparing admissible modulo-30 structure before prime/composite labeling.


## Goals

1. Detect repo root and create standard output directories.
2. Load MRL foundations from Notebook 00 when available.
3. Generate interval samples for modulo-30 lanes.
4. Isolate lane `01`.
5. Compare lane `01` against all eight admissible lanes.
6. Export CSV, JSON, figures, and Markdown report.
7. Create a Colab-downloadable output zip.


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/mod30-residue-lanes"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "src" / "mod30_residue_lanes").exists() or (c / "notebooks").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [NOTEBOOKS_DIR, RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)
print("REPORTS_DIR:", REPORTS_DIR)


## Load Notebook 00 foundation output

Preferred input:

```text
results/notebook00_mrl_foundations.json
```

If unavailable, this notebook uses the standard MRL defaults.


In [ ]:
foundation_path = RESULTS_DIR / "notebook00_mrl_foundations.json"

if foundation_path.exists():
    foundation = json.loads(foundation_path.read_text())
    MODULUS = foundation["modulus"]
    ADMISSIBLE_RESIDUES = foundation["admissible_residues"]
    print("Loaded foundation:", foundation_path)
else:
    MODULUS = 30
    ADMISSIBLE_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]
    print("Notebook 00 foundation not found; using defaults.")

TARGET_LANE = 1
LANE_LABEL = f"{TARGET_LANE:02d}"

print("MODULUS:", MODULUS)
print("ADMISSIBLE_RESIDUES:", ADMISSIBLE_RESIDUES)
print("TARGET_LANE:", LANE_LABEL)


## Generate interval sample

Default interval: integers `1..3000`, giving 100 complete modulo-30 cycles.


In [ ]:
N_MAX = 3000
values = np.arange(1, N_MAX + 1)

df = pd.DataFrame({
    "n": values,
    "residue": values % MODULUS,
})

df["is_admissible"] = df["residue"].isin(ADMISSIBLE_RESIDUES)
df["is_target_lane"] = df["residue"] == TARGET_LANE
df["cycle"] = (df["n"] - 1) // MODULUS

admissible_df = df[df["is_admissible"]].copy()
lane01_df = df[df["is_target_lane"]].copy()

print("Total values:", len(df))
print("Admissible values:", len(admissible_df))
print("Lane 01 values:", len(lane01_df))

lane01_df.head()


## Lane counts and densities

In [ ]:
lane_counts = (
    admissible_df
    .groupby("residue")
    .size()
    .reindex(ADMISSIBLE_RESIDUES)
    .rename("count")
    .reset_index()
)

lane_counts["lane_label"] = lane_counts["residue"].map(lambda x: f"{x:02d}")
lane_counts["density_in_full_interval"] = lane_counts["count"] / len(df)
lane_counts["density_among_admissible"] = lane_counts["count"] / len(admissible_df)
lane_counts["is_target_lane"] = lane_counts["residue"] == TARGET_LANE

lane_counts_csv = RESULTS_DIR / "notebook01_lane_counts.csv"
lane_counts.to_csv(lane_counts_csv, index=False)

print(lane_counts_csv)
lane_counts


## Lane 01 spacing structure

Within complete modulo-30 cycles, lane `01` repeats every 30 integers.


In [ ]:
lane01_values = lane01_df["n"].to_numpy()
lane01_spacing = np.diff(lane01_values)

spacing_df = pd.DataFrame({
    "from_n": lane01_values[:-1],
    "to_n": lane01_values[1:],
    "spacing": lane01_spacing,
})

spacing_csv = RESULTS_DIR / "notebook01_lane01_spacing.csv"
spacing_df.to_csv(spacing_csv, index=False)

spacing_summary = spacing_df["spacing"].describe().to_frame().T
spacing_summary["unique_spacings"] = ", ".join(map(str, sorted(spacing_df["spacing"].unique())))

print(spacing_csv)
spacing_summary


## Interval-window lane vectors

Each window becomes an eight-entry vector ordered by:

```text
01, 07, 11, 13, 17, 19, 23, 29
```


In [ ]:
WINDOW_SIZE = 300

window_rows = []

for start in range(1, N_MAX + 1, WINDOW_SIZE):
    stop = min(start + WINDOW_SIZE - 1, N_MAX)
    window = df[(df["n"] >= start) & (df["n"] <= stop)]
    counts = window[window["is_admissible"]].groupby("residue").size().reindex(ADMISSIBLE_RESIDUES, fill_value=0)

    row = {
        "window_start": start,
        "window_stop": stop,
        "window_size": len(window),
    }
    for residue, count in counts.items():
        row[f"lane_{residue:02d}"] = int(count)
    row["lane_01_share"] = int(counts.loc[1]) / max(1, int(counts.sum()))
    window_rows.append(row)

window_vectors = pd.DataFrame(window_rows)

window_vectors_csv = RESULTS_DIR / "notebook01_window_lane_vectors.csv"
window_vectors.to_csv(window_vectors_csv, index=False)

print(window_vectors_csv)
window_vectors.head()


## Similarity of lane 01 to all lanes

For this first notebook, similarity is computed from window-count trajectories.


In [ ]:
def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

lane_cols = [f"lane_{r:02d}" for r in ADMISSIBLE_RESIDUES]
target_vector = window_vectors["lane_01"].to_numpy()

similarity_rows = []
for r in ADMISSIBLE_RESIDUES:
    col = f"lane_{r:02d}"
    sim = cosine_similarity(target_vector, window_vectors[col].to_numpy())
    similarity_rows.append({
        "target_lane": "01",
        "comparison_lane": f"{r:02d}",
        "cosine_similarity": sim,
    })

similarity_df = pd.DataFrame(similarity_rows)

similarity_csv = RESULTS_DIR / "notebook01_lane01_similarity.csv"
similarity_df.to_csv(similarity_csv, index=False)

print(similarity_csv)
similarity_df


## Summary exports

In [ ]:
summary = {
    "repo": "mod30-residue-lanes",
    "notebook": "01_lane_residue_1",
    "modulus": MODULUS,
    "target_lane": TARGET_LANE,
    "target_lane_label": LANE_LABEL,
    "admissible_residues": ADMISSIBLE_RESIDUES,
    "n_max": N_MAX,
    "window_size": WINDOW_SIZE,
    "total_values": int(len(df)),
    "admissible_values": int(len(admissible_df)),
    "target_lane_values": int(len(lane01_df)),
    "target_lane_density_full_interval": float(len(lane01_df) / len(df)),
    "target_lane_density_admissible": float(len(lane01_df) / len(admissible_df)),
    "unique_lane01_spacings": sorted([int(x) for x in spacing_df["spacing"].unique()]),
    "mean_lane01_spacing": float(spacing_df["spacing"].mean()),
    "constraint_view": "Lane 01 acts as the anchor lane for comparing admissible modulo-30 structure before prime/composite labeling.",
}

json_path = RESULTS_DIR / "notebook01_lane_residue_1_summary.json"
json_path.write_text(json.dumps(summary, indent=2))

print(json_path)
print(json.dumps(summary, indent=2))


## Figures

In [ ]:
figure_names = []

def savefig(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    figure_names.append(name)
    print(path)

plt.figure(figsize=(8, 4))
plt.bar(lane_counts["lane_label"], lane_counts["count"])
plt.xlabel("Residue lane")
plt.ylabel("Count")
plt.title("Notebook 01 — Admissible Lane Counts")
savefig("notebook01_admissible_lane_counts.png")

plt.figure(figsize=(10, 4))
plt.scatter(lane01_df["cycle"], lane01_df["n"], s=12)
plt.xlabel("Modulo-30 cycle")
plt.ylabel("n")
plt.title("Notebook 01 — Lane 01 Positions Across Cycles")
savefig("notebook01_lane01_positions.png")

plt.figure(figsize=(7, 4))
plt.hist(spacing_df["spacing"], bins=range(0, 62, 2))
plt.xlabel("Spacing")
plt.ylabel("Frequency")
plt.title("Notebook 01 — Lane 01 Spacing Distribution")
savefig("notebook01_lane01_spacing_distribution.png")

matrix = window_vectors[lane_cols].to_numpy()
plt.figure(figsize=(10, 5))
plt.imshow(matrix, aspect="auto")
plt.xticks(range(len(lane_cols)), [c.replace("lane_", "") for c in lane_cols])
plt.yticks(range(len(window_vectors)), window_vectors["window_start"])
plt.xlabel("Residue lane")
plt.ylabel("Window start")
plt.colorbar(label="Count")
plt.title("Notebook 01 — Window Lane-Count Vectors")
savefig("notebook01_window_lane_vectors.png")

plt.figure(figsize=(8, 4))
plt.bar(similarity_df["comparison_lane"], similarity_df["cosine_similarity"])
plt.xlabel("Comparison lane")
plt.ylabel("Cosine similarity to lane 01")
plt.ylim(0, 1.05)
plt.title("Notebook 01 — Lane 01 Similarity Across Windows")
savefig("notebook01_lane01_similarity.png")


## Build Markdown report

In [ ]:
report_path = REPORTS_DIR / "report_01_lane_residue_1.md"

output_links = "\n".join([
    '- Lane counts CSV: <a href="../results/notebook01_lane_counts.csv">`results/notebook01_lane_counts.csv`</a>',
    '- Lane 01 spacing CSV: <a href="../results/notebook01_lane01_spacing.csv">`results/notebook01_lane01_spacing.csv`</a>',
    '- Window lane vectors CSV: <a href="../results/notebook01_window_lane_vectors.csv">`results/notebook01_window_lane_vectors.csv`</a>',
    '- Lane 01 similarity CSV: <a href="../results/notebook01_lane01_similarity.csv">`results/notebook01_lane01_similarity.csv`</a>',
    '- Summary JSON: <a href="../results/notebook01_lane_residue_1_summary.json">`results/notebook01_lane_residue_1_summary.json`</a>',
] + [
    f'- Figure: <a href="../figures/{name}">`figures/{name}`</a>'
    for name in figure_names
])

report = f"""# Report 01 — Lane Residue 1

Notebook 01 begins lane-specific MRL analysis with residue lane `01`.

Constraint view:

> Lane 01 acts as the anchor lane for comparing admissible modulo-30 structure before prime/composite labeling.

## Generated outputs

{output_links}

## Summary

| Metric | Value |
|---|---:|
| Modulus | {MODULUS} |
| Target lane | {LANE_LABEL} |
| Total interval values | {len(df)} |
| Admissible values | {len(admissible_df)} |
| Lane 01 values | {len(lane01_df)} |
| Lane 01 density in full interval | {len(lane01_df) / len(df):.6f} |
| Lane 01 density among admissible values | {len(lane01_df) / len(admissible_df):.6f} |
| Mean lane 01 spacing | {spacing_df["spacing"].mean():.6f} |

## Lane counts

{lane_counts.to_markdown(index=False)}

## Lane 01 spacing summary

{spacing_summary.to_markdown(index=False)}

## Lane 01 similarity

{similarity_df.to_markdown(index=False)}

## Interpretation

- Lane `01` is one of eight admissible modulo-30 residue lanes.
- Across complete modulo-30 cycles, each admissible lane appears with equal count.
- Lane `01` repeats every 30 integers in the full integer stream.
- Window vectors give a reusable eight-lane representation for later MRL, CGCS, and correlation analysis.

## Next step

Notebook 07 can compare lane `07` against lane `01` and begin asymmetric-lift analysis across the residue spine.
"""

report_path.write_text(report)
print(report_path)


## Report preview

In [ ]:
print(report_path.read_text()[:5000])


## Render generated figures in notebook

In [ ]:
from IPython.display import Image, display, Markdown

for name in figure_names:
    display(Markdown(f"### `{name}`"))
    display(Image(filename=str(FIGURES_DIR / name)))


## Create output zip

In [ ]:
EXPORT_NAME = "notebook01_lane_residue_1_outputs.zip"
export_path = REPO_ROOT / EXPORT_NAME

with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.glob("notebook01_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in FIGURES_DIR.glob("notebook01_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in REPORTS_DIR.glob("report_01_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))

print(export_path)


## Optional Colab download

Uncomment and run this cell in Colab to download generated outputs.


In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook01_lane_residue_1_outputs.zip"
# export_path = REPO_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook01_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#         for p in folder.glob("report_01_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))
